# 📘 Chapter 04. 추론(Inference)

> **학습 목표**
> - 학습(Training)과 추론(Inference)의 차이를 이해한다.
> - 저장된 모델 가중치를 불러와 추론에 사용할 수 있다.
> - 직접 손글씨를 그려 모델이 예측하는 과정을 체험한다.

---

### 📌 학습(Training) vs 추론(Inference)

| 구분 | 학습 (Training) | 추론 (Inference) |
|------|-----------------|------------------|
| **목적** | 모델의 가중치를 최적화 | 학습된 모델로 새로운 데이터를 예측 |
| **데이터** | 학습 데이터 (정답 라벨 포함) | 새로운 데이터 (정답 없음) |
| **기울기 계산** | ✅ 필요 (Backward Pass) | ❌ 불필요 (`torch.no_grad()`) |
| **모드** | `model.train()` | `model.eval()` |
| **속도** | 상대적으로 느림 | 빠름 (기울기 계산 없으므로) |

```
오늘의 흐름:

  ① 저장된 모델 가중치 불러오기 (.pth)
      ▼
  ② 손글씨 그리기 (ipycanvas)
      ▼
  ③ 이미지 전처리 (280×280 → 28×28, 정규화)
      ▼
  ④ 모델 추론 (Forward Pass only)
      ▼
  ⑤ 결과 확인 (예측 숫자 + 확률)
```

---
## 1. 환경 설정

필요한 라이브러리를 불러옵니다.

> 💡 `ipycanvas`가 설치되어 있지 않다면, 아래 셀의 주석을 해제하여 설치하세요.

In [ ]:
# ipycanvas 설치 (필요한 경우 주석 해제)
# !pip install ipycanvas

In [ ]:
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from ipycanvas import Canvas
import ipywidgets as widgets
from IPython.display import display

# GPU 사용 가능 여부 확인
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'사용 디바이스: {device}')

---
## 2. ⭐ 학습된 모델 불러오기 (핵심 실습)

### 📌 모델 불러오기의 2단계

```
Step 1: 모델 구조 정의 (Chapter 03과 동일한 클래스)
Step 2: 저장된 가중치(.pth)를 모델에 로드
```

모델을 불러올 때는 **구조(Architecture)**와 **가중치(Weights)**가 모두 필요합니다.
- 구조: `ShallowCNN` 클래스를 다시 정의 (Chapter 03과 동일)
- 가중치: `torch.load()`로 `.pth` 파일에서 불러오기

### 📌 `model.eval()`을 호출하는 이유
- 추론 시에는 Dropout, BatchNorm 등이 **학습 모드가 아닌 평가 모드**로 동작해야 합니다.
- 우리 모델에는 해당 레이어가 없지만, 좋은 습관으로 항상 호출합니다.

In [ ]:
# ============================================================
# Step 1: 모델 구조 정의 (Chapter 03과 동일)
# ============================================================
class ShallowCNN(nn.Module):
    def __init__(self):
        super(ShallowCNN, self).__init__()

        # Block 1: Conv → ReLU → MaxPool
        self.conv1 = nn.Conv2d(in_channels=1, out_channels=16, kernel_size=3, padding=1)
        self.relu1 = nn.ReLU()
        self.pool1 = nn.MaxPool2d(kernel_size=2)

        # Block 2: Conv → ReLU → MaxPool
        self.conv2 = nn.Conv2d(in_channels=16, out_channels=32, kernel_size=3, padding=1)
        self.relu2 = nn.ReLU()
        self.pool2 = nn.MaxPool2d(kernel_size=2)

        # Classifier
        self.flatten = nn.Flatten()
        self.fc = nn.Linear(in_features=32 * 7 * 7, out_features=10)

    def forward(self, x):
        x = self.conv1(x)
        x = self.relu1(x)
        x = self.pool1(x)

        x = self.conv2(x)
        x = self.relu2(x)
        x = self.pool2(x)

        x = self.flatten(x)
        x = self.fc(x)
        return x

In [ ]:
# ============================================================
# Step 2: 저장된 가중치 불러오기
# ============================================================

# 모델 생성
model = ShallowCNN().to(device)

# ============================================================
# [실습] 아래 주석을 해제하여 가중치를 로드하세요.
# ============================================================

# 저장된 가중치 파일 경로 (Chapter 03에서 저장한 파일)
MODEL_PATH = '../chapter 03. Shallow CNN 모델 및 학습 Loop 구현/mnist_shallow_cnn.pth'

# 가중치 로드
# model.load_state_dict(torch.load(MODEL_PATH, map_location=device))

# 추론 모드로 전환 (학습 모드 → 평가 모드)
# model.eval()

print('✅ 모델 로드 완료!')
print(model)

---
## 3. ✏️ 손글씨 그리기

### 📌 ipycanvas를 활용한 드로잉

`ipycanvas`는 Jupyter Notebook 내에서 HTML5 Canvas를 사용할 수 있게 해주는 위젯입니다.
별도의 파일 업로드 없이, **노트북 안에서 직접 마우스로 숫자를 그릴 수 있습니다.**

### 📌 MNIST 데이터와 동일한 형식으로 그리기
- **검정 배경에 흰색 글씨**: MNIST 데이터셋의 형식과 동일
- **캔버스 크기 280×280**: 28×28로 축소하기 쉽도록 10배 크기로 설정
- **두꺼운 선 (20px)**: 축소 후에도 글씨가 잘 보이도록

아래 셀을 실행하면 드로잉 캔버스가 나타납니다. **마우스로 숫자를 크게 그려보세요!**

In [ ]:
# ============================================================
# 드로잉 캔버스 생성
# ============================================================

CANVAS_SIZE = 280  # 28 × 10 (MNIST 크기의 10배)

canvas = Canvas(width=CANVAS_SIZE, height=CANVAS_SIZE)

# 검정 배경 (MNIST 형식: 배경=검정)
canvas.fill_style = 'black'
canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)

# 흰색 펜 설정 (MNIST 형식: 글씨=흰색)
canvas.stroke_style = 'white'
canvas.line_width = 20
canvas.line_cap = 'round'
canvas.line_join = 'round'

# --- 마우스 이벤트 핸들러 ---
is_drawing = False
last_x, last_y = 0, 0

def on_mouse_down(x, y):
    global is_drawing, last_x, last_y
    is_drawing = True
    last_x, last_y = x, y

def on_mouse_move(x, y):
    global last_x, last_y
    if is_drawing:
        canvas.stroke_line(last_x, last_y, x, y)
        last_x, last_y = x, y

def on_mouse_up(x, y):
    global is_drawing
    is_drawing = False

canvas.on_mouse_down(on_mouse_down)
canvas.on_mouse_move(on_mouse_move)
canvas.on_mouse_up(on_mouse_up)

# --- 지우기 버튼 ---
def clear_canvas(_):
    canvas.fill_style = 'black'
    canvas.fill_rect(0, 0, CANVAS_SIZE, CANVAS_SIZE)
    canvas.stroke_style = 'white'
    print('🗑️ 캔버스를 지웠습니다. 다시 그려주세요!')

clear_btn = widgets.Button(description='🗑️ 지우기', button_style='warning')
clear_btn.on_click(clear_canvas)

# --- 화면에 표시 ---
print('✏️ 아래 검정 캔버스에 숫자를 크게 그려주세요! (마우스 드래그)')
display(canvas)
display(clear_btn)

### 💾 그린 이미지 저장

그림을 다 그렸다면, 아래 셀을 실행하여 캔버스를 이미지 파일로 저장합니다.

In [ ]:
# 캔버스를 PNG 파일로 저장
DRAWN_IMAGE_PATH = 'my_digit.png'
canvas.to_file(DRAWN_IMAGE_PATH)

print(f'✅ 이미지 저장 완료: {DRAWN_IMAGE_PATH}')
print('   다음 셀에서 이미지를 전처리하고 추론합니다.')

---
## 4. ⭐ 이미지 전처리 (핵심 실습)

### 📌 왜 전처리가 필요할까?

우리가 캔버스에 그린 이미지는 **280×280 크기의 RGBA 컬러 이미지**입니다.
하지만 모델이 학습한 MNIST 데이터는 **28×28 크기의 흑백 이미지 (0~1 범위)**입니다.

따라서 모델에 입력하기 전에 **동일한 형태로 변환**해주어야 합니다.

```
전처리 파이프라인:

  캔버스 이미지 (280×280, RGBA)
      │
      ▼  ① 흑백(Grayscale) 변환
  흑백 이미지 (280×280, 1채널)
      │
      ▼  ② 28×28로 리사이즈
  축소 이미지 (28×28, 1채널)
      │
      ▼  ③ 0~255 → 0~1 정규화
  정규화 이미지 (28×28, 값: 0.0~1.0)
      │
      ▼  ④ PyTorch Tensor 변환 + 차원 추가
  모델 입력 텐서 (1, 1, 28, 28)
```

### 📝 실습 안내
아래 코드의 주석을 해제하여 전처리 파이프라인을 완성하세요!

In [ ]:
def preprocess_image(image_path):
    """
    캔버스에서 저장한 이미지를 모델 입력 형태로 전처리합니다.

    Args:
        image_path: 저장된 이미지 파일 경로

    Returns:
        tensor: (1, 1, 28, 28) 형태의 PyTorch Tensor
        img_28: 28×28 NumPy 배열 (시각화용)
    """
    # ============================================================
    # [실습] 아래 주석을 해제하여 전처리 파이프라인을 완성하세요.
    # ============================================================

    # ① PIL로 이미지 로드 및 흑백 변환
    # img = Image.open(image_path).convert('L')   # 'L' = Grayscale (흑백)

    # ② 28×28로 리사이즈 (MNIST와 동일한 크기)
    # img = img.resize((28, 28), Image.LANCZOS)   # LANCZOS: 고품질 축소 알고리즘

    # ③ NumPy 배열로 변환 후 0~1로 정규화
    # img_array = np.array(img, dtype=np.float32) / 255.0

    # ④ PyTorch Tensor로 변환 + 배치/채널 차원 추가
    #    (28, 28) → (1, 1, 28, 28)  = (Batch, Channel, Height, Width)
    # tensor = torch.tensor(img_array).unsqueeze(0).unsqueeze(0)

    return tensor.to(device), img_array


# 전처리 실행
input_tensor, img_28 = preprocess_image(DRAWN_IMAGE_PATH)

print(f'전처리 후 텐서 Shape: {input_tensor.shape}')    # (1, 1, 28, 28)
print(f'전처리 후 값 범위: {input_tensor.min():.2f} ~ {input_tensor.max():.2f}')

### 🔍 전처리 결과 확인

원본(280×280)과 전처리된 이미지(28×28)를 나란히 비교합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(8, 4))

# 원본 이미지 (280×280)
original = Image.open(DRAWN_IMAGE_PATH).convert('L')
axes[0].imshow(np.array(original), cmap='gray')
axes[0].set_title(f'원본 ({original.size[0]}×{original.size[1]})', fontsize=12)
axes[0].axis('off')

# 전처리된 이미지 (28×28) — 모델이 실제로 보는 이미지
axes[1].imshow(img_28, cmap='gray')
axes[1].set_title(f'전처리 후 (28×28)', fontsize=12)
axes[1].axis('off')

plt.suptitle('모델이 보는 이미지', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 5. ⭐ 모델 추론 (핵심 실습)

### 📌 추론 과정

학습 때와 달리, 추론에서는 **Forward Pass만** 수행합니다.

```
학습 (Training):     Forward → Loss → Backward → Update  (4단계)
추론 (Inference):    Forward → 결과 확인                   (1단계)
```

- `torch.no_grad()`: 기울기 계산을 하지 않음 → **메모리 절약, 속도 향상**
- `torch.softmax()`: 모델의 원시 출력(logits)을 **확률(0~1)**로 변환

### 📝 실습 안내
아래 코드의 주석을 해제하여 추론을 수행하세요!

In [ ]:
# ============================================================
# [실습] 아래 주석을 해제하여 추론을 완성하세요.
# ============================================================

# 기울기 계산을 비활성화 (추론 시 불필요)
# with torch.no_grad():
#     # Forward Pass: 모델에 전처리된 이미지를 넣어 예측
#     output = model(input_tensor)
#
#     # Softmax: 원시 점수(logits) → 확률(0~1)로 변환
#     probabilities = torch.softmax(output, dim=1)
#
#     # 가장 높은 확률의 클래스 = 예측 결과
#     predicted_class = probabilities.argmax(dim=1).item()
#     confidence = probabilities[0][predicted_class].item() * 100

print(f'🔮 예측 결과: {predicted_class}')
print(f'📊 확신도: {confidence:.1f}%')

### 📊 클래스별 확률 시각화

모델이 각 숫자(0~9)에 대해 몇 %의 확률로 예측했는지 막대 그래프로 확인합니다.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 왼쪽: 입력 이미지
axes[0].imshow(img_28, cmap='gray')
axes[0].set_title(f'입력 이미지 (28×28)', fontsize=12)
axes[0].axis('off')

# 오른쪽: 클래스별 확률 막대 그래프
probs = probabilities[0].cpu().numpy()
classes = np.arange(10)
colors = ['#e74c3c' if i == predicted_class else '#bdc3c7' for i in classes]

bars = axes[1].bar(classes, probs * 100, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('숫자 (Class)', fontsize=11)
axes[1].set_ylabel('확률 (%)', fontsize=11)
axes[1].set_title(f'예측: "{predicted_class}" ({confidence:.1f}%)', fontsize=14, fontweight='bold')
axes[1].set_xticks(classes)
axes[1].set_ylim(0, 105)
axes[1].grid(axis='y', alpha=0.3)

# 예측 클래스 막대 위에 확률 표시
for i, (bar, p) in enumerate(zip(bars, probs)):
    if p > 0.01:
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                     f'{p*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('🔮 모델 추론 결과', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 6. 🔄 다시 그려서 추론하기

다른 숫자를 그려서 다시 추론해보고 싶다면:
1. **위로 스크롤**하여 Section 3의 캔버스로 이동
2. `🗑️ 지우기` 버튼을 눌러 캔버스 초기화
3. 새로운 숫자를 그리기
4. Section 3의 **저장 셀** ~ Section 5의 **추론 셀**까지 순서대로 재실행

또는 아래의 **올인원 셀**을 사용하세요. 캔버스에 새 숫자를 그린 후 아래 셀만 실행하면 저장 → 전처리 → 추론 → 시각화를 한번에 수행합니다.

In [ ]:
# ============================================================
# 올인원: 저장 → 전처리 → 추론 → 시각화
# ============================================================

# 1. 캔버스 저장
canvas.to_file(DRAWN_IMAGE_PATH)

# 2. 전처리
input_tensor, img_28 = preprocess_image(DRAWN_IMAGE_PATH)

# 3. 추론
with torch.no_grad():
    output = model(input_tensor)
    probabilities = torch.softmax(output, dim=1)
    predicted_class = probabilities.argmax(dim=1).item()
    confidence = probabilities[0][predicted_class].item() * 100

# 4. 시각화
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

axes[0].imshow(img_28, cmap='gray')
axes[0].set_title('입력 이미지 (28×28)', fontsize=12)
axes[0].axis('off')

probs = probabilities[0].cpu().numpy()
classes = np.arange(10)
colors = ['#e74c3c' if i == predicted_class else '#bdc3c7' for i in classes]
bars = axes[1].bar(classes, probs * 100, color=colors, edgecolor='white', linewidth=0.5)
axes[1].set_xlabel('숫자 (Class)', fontsize=11)
axes[1].set_ylabel('확률 (%)', fontsize=11)
axes[1].set_title(f'예측: "{predicted_class}" ({confidence:.1f}%)', fontsize=14, fontweight='bold')
axes[1].set_xticks(classes)
axes[1].set_ylim(0, 105)
axes[1].grid(axis='y', alpha=0.3)
for i, (bar, p) in enumerate(zip(bars, probs)):
    if p > 0.01:
        axes[1].text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 1,
                     f'{p*100:.1f}%', ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.suptitle('🔮 모델 추론 결과', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()